# RegRAG-VN: Colab Execution Runner

This notebook runs the 4-bit quantized small LLMs across 3 configurations (Closed-book, RAG-BM25, RAG-Dense) on the 100 gold legal questions.
It incorporates Google Drive checkpoint save and restore cells for session resilience.

In [ ]:
# [CELL 1]: Environment Setup & Dependencies
!pip install -q transformers accelerate bitsandbytes sentence-transformers rank-bm25 pyvi
!git clone https://github.com/alitonia/rag_eval.git /content/rag_eval || (cd /content/rag_eval && git pull)
%cd /content/rag_eval

In [ ]:
# [CELL 2]: Google Drive Mount & Checkpoint Restore
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/regrag_vn_checkpoints'
LOCAL_RESULTS_DIR = '/content/rag_eval/results'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOCAL_RESULTS_DIR, exist_ok=True)

# Restore existing checkpoint if present
checkpoint_file = os.path.join(DRIVE_CHECKPOINT_DIR, 'evaluations.json')
if os.path.exists(checkpoint_file):
    shutil.copy(checkpoint_file, os.path.join(LOCAL_RESULTS_DIR, 'evaluations.json'))
    print(f"[RESTORED] Checkpoint loaded from {checkpoint_file}")
else:
    print("[FRESH RUN] No prior checkpoint found in Google Drive. Starting fresh.")

In [ ]:
# [CELL 3]: Load Storage and Check Completed Runs
from regrag.storage.file_repo import FileResultRepository

repo = FileResultRepository(LOCAL_RESULTS_DIR)
existing_evals = repo.list_evaluations()
completed_keys = set((e.question_id, e.model_name, e.retrieval_mode) for e in existing_evals)
print(f"Total completed evaluations already restored: {len(completed_keys)} / 900")

In [ ]:
# [CELL 4]: Save Checkpoint to Google Drive
def sync_checkpoint_to_drive():
    local_json = os.path.join(LOCAL_RESULTS_DIR, 'evaluations.json')
    local_csv = os.path.join(LOCAL_RESULTS_DIR, 'evaluations.csv')
    if os.path.exists(local_json):
        shutil.copy(local_json, os.path.join(DRIVE_CHECKPOINT_DIR, 'evaluations.json'))
    if os.path.exists(local_csv):
        shutil.copy(local_csv, os.path.join(DRIVE_CHECKPOINT_DIR, 'evaluations.csv'))
    print(f"[CHECKPOINT SYNCED] Saved to {DRIVE_CHECKPOINT_DIR}")

sync_checkpoint_to_drive()